# YOLO26 비전 태스크: Detection을 넘어서

YOLO26이 지원하는 5가지 비전 태스크를 실습합니다.

| 태스크 | 모델 | 핵심 질문 |
|--------|------|-----------|
| **Detection** | `yolo26n.pt` | 무엇이 어디에? |
| **Segmentation** | `yolo26n-seg.pt` | 객체의 정확한 윤곽은? |
| **Pose** | `yolo26n-pose.pt` | 사람의 자세는? |
| **OBB** | `yolo26n-obb.pt` | 객체의 방향은? |
| **Classification** | `yolo26n-cls.pt` | 이 이미지는 무엇? |

---

## 0. 환경 설정

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 공통 임포트
from ultralytics import YOLO
from IPython.display import Image, display
import matplotlib.pyplot as plt
import numpy as np
import cv2
from PIL import Image as PILImage

In [ ]:
# 테스트 이미지 다운로드
import urllib.request
import os

os.makedirs("test_images", exist_ok=True)

urls = {
    "bus.jpg": "https://ultralytics.com/images/bus.jpg",
    "zidane.jpg": "https://ultralytics.com/images/zidane.jpg",
}

for name, url in urls.items():
    path = f"test_images/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
        print(f"Downloaded: {name}")
    else:
        print(f"Already exists: {name}")

---
## 1. Detection vs Segmentation 비교

같은 이미지에 Detection과 Segmentation을 적용하여 결과가 어떻게 다른지 비교합니다.

- **Detection**: 사각형 박스 ("여기에 버스가 있다")
- **Segmentation**: 픽셀 마스크 ("버스의 정확한 윤곽은 이렇다")

### 1-1. Detection 결과

In [ ]:
# Detection 모델 로드 및 추론
det_model = YOLO("yolo26n.pt")
det_results = det_model("test_images/bus.jpg")

# 결과 시각화
det_plot = det_results[0].plot()
plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(det_plot, cv2.COLOR_BGR2RGB))
plt.title("Detection: Bounding Boxes")
plt.axis("off")
plt.show()

In [ ]:
# Detection 결과 데이터 확인
det_r = det_results[0]
print(f"탐지된 객체 수: {len(det_r.boxes)}")
print(f"\n--- boxes 구조 ---")
print(f"boxes.xyxy (좌표): shape = {det_r.boxes.xyxy.shape}")
print(f"boxes.conf (신뢰도): shape = {det_r.boxes.conf.shape}")
print(f"boxes.cls  (클래스): shape = {det_r.boxes.cls.shape}")

print(f"\n--- 탐지 결과 ---")
for i, (box, conf, cls) in enumerate(zip(
    det_r.boxes.xyxy.cpu().numpy(),
    det_r.boxes.conf.cpu().numpy(),
    det_r.boxes.cls.cpu().numpy().astype(int)
)):
    print(f"  [{i}] {det_r.names[cls]:>10s} {conf:.2f}  box=({box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f})")

### 1-2. Segmentation 결과

In [ ]:
# Segmentation 모델 로드 및 추론
seg_model = YOLO("yolo26n-seg.pt")
seg_results = seg_model("test_images/bus.jpg")

# 결과 시각화
seg_plot = seg_results[0].plot()
plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(seg_plot, cv2.COLOR_BGR2RGB))
plt.title("Segmentation: Pixel Masks")
plt.axis("off")
plt.show()

In [ ]:
# Segmentation 결과 데이터 확인
seg_r = seg_results[0]
print(f"탐지된 객체 수: {len(seg_r.boxes)}")
print(f"\n--- masks 구조 ---")
print(f"masks.data shape: {seg_r.masks.data.shape}")  # (N, H, W)
print(f"masks.data dtype: {seg_r.masks.data.dtype}")
print(f"\n→ Detection에는 없는 masks 데이터가 추가됨!")
print(f"→ 각 객체마다 (H, W) 크기의 이진 마스크가 존재")

### 1-3. 나란히 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(cv2.cvtColor(det_plot, cv2.COLOR_BGR2RGB))
axes[0].set_title("Detection (Bounding Box)", fontsize=16)
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(seg_plot, cv2.COLOR_BGR2RGB))
axes[1].set_title("Segmentation (Pixel Mask)", fontsize=16)
axes[1].axis("off")

plt.suptitle("같은 이미지, 다른 태스크", fontsize=20, y=1.02)
plt.tight_layout()
plt.show()

### 1-4. 마스크 활용: 객체별 면적 계산

In [ ]:
# 각 객체의 마스크 면적(픽셀 수) 계산
masks = seg_r.masks.data.cpu().numpy()  # (N, H, W)
orig_h, orig_w = seg_r.orig_img.shape[:2]

print(f"원본 이미지 크기: {orig_w} x {orig_h} = {orig_w * orig_h:,} 픽셀\n")
print(f"{'객체':>10s} {'신뢰도':>8s} {'마스크 픽셀':>12s} {'비율':>8s}")
print("-" * 45)

for i, (mask, conf, cls) in enumerate(zip(
    masks,
    seg_r.boxes.conf.cpu().numpy(),
    seg_r.boxes.cls.cpu().numpy().astype(int)
)):
    # 마스크를 원본 크기로 리사이즈
    mask_resized = cv2.resize(mask, (orig_w, orig_h))
    pixel_count = int((mask_resized > 0.5).sum())
    ratio = pixel_count / (orig_w * orig_h) * 100
    print(f"{seg_r.names[cls]:>10s} {conf:>8.2f} {pixel_count:>12,} {ratio:>7.1f}%")

### 1-5. 마스크 활용: 배경 제거

In [ ]:
# 첫 번째 객체의 마스크로 배경 제거
orig_img = seg_r.orig_img  # BGR
orig_h, orig_w = orig_img.shape[:2]

# 가장 큰 마스크 찾기
mask_areas = [(cv2.resize(m, (orig_w, orig_h)) > 0.5).sum() for m in masks]
largest_idx = np.argmax(mask_areas)
largest_name = seg_r.names[int(seg_r.boxes.cls[largest_idx])]

# 마스크 적용
mask = cv2.resize(masks[largest_idx], (orig_w, orig_h))
mask_bool = mask > 0.5

# 배경을 흰색으로
result = orig_img.copy()
result[~mask_bool] = [255, 255, 255]  # 배경을 흰색으로

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title(f"Mask: {largest_name}")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Background Removed: {largest_name}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 메모리 정리
del det_model, seg_model, det_results, seg_results
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 2. Pose Estimation — AI 자세 분석

사람의 17개 관절(키포인트)을 추출하고, 관절 각도를 계산합니다.

| 인덱스 | 키포인트 | 인덱스 | 키포인트 |
|--------|---------|--------|----------|
| 0 | 코 | 9 | 왼손목 |
| 1 | 왼눈 | 10 | 오른손목 |
| 2 | 오른눈 | 11 | 왼엉덩이 |
| 3 | 왼귀 | 12 | 오른엉덩이 |
| 4 | 오른귀 | 13 | 왼무릎 |
| 5 | 왼어깨 | 14 | 오른무릎 |
| 6 | 오른어깨 | 15 | 왼발목 |
| 7 | 왼팔꿈치 | 16 | 오른발목 |
| 8 | 오른팔꿈치 | | |

### 2-1. 이미지에서 Pose 추출

In [ ]:
pose_model = YOLO("yolo26n-pose.pt")
pose_results = pose_model("test_images/zidane.jpg")

# 결과 시각화
pose_plot = pose_results[0].plot()
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(pose_plot, cv2.COLOR_BGR2RGB))
plt.title("Pose Estimation: 17 Keypoints", fontsize=16)
plt.axis("off")
plt.show()

In [ ]:
# 키포인트 데이터 구조 확인
pose_r = pose_results[0]
kpts = pose_r.keypoints

print(f"감지된 사람 수: {len(kpts)}")
print(f"keypoints.xy shape: {kpts.xy.shape}  → (사람 수, 17개 관절, x/y 좌표)")
print(f"keypoints.conf shape: {kpts.conf.shape}  → (사람 수, 17개 관절 신뢰도)")

# 키포인트 이름 매핑
KPT_NAMES = [
    "코", "왼눈", "오른눈", "왼귀", "오른귀",
    "왼어깨", "오른어깨", "왼팔꿈치", "오른팔꿈치",
    "왼손목", "오른손목", "왼엉덩이", "오른엉덩이",
    "왼무릎", "오른무릎", "왼발목", "오른발목"
]

# 첫 번째 사람의 키포인트 출력
print(f"\n--- 첫 번째 사람의 키포인트 ---")
xy = kpts.xy[0].cpu().numpy()
conf = kpts.conf[0].cpu().numpy()

for i, (name, (x, y), c) in enumerate(zip(KPT_NAMES, xy, conf)):
    status = "✓" if c > 0.5 else "✗"
    print(f"  [{i:2d}] {name:<6s}  ({x:6.1f}, {y:6.1f})  conf={c:.2f} {status}")

### 2-2. 관절 각도 계산

세 점(어깨, 팔꿈치, 손목)으로 팔의 굽힘 각도를 계산합니다.

In [ ]:
def calc_angle(a, b, c):
    """
    세 점(a, b, c)에서 b를 꼭짓점으로 하는 각도를 계산합니다.
    
    Args:
        a, b, c: (x, y) 좌표 배열
    Returns:
        각도 (도 단위, 0~180)
    """
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1, 1)))


# 첫 번째 사람의 키포인트
xy = kpts.xy[0].cpu().numpy()

# 왼쪽 팔: 어깨(5) - 팔꿈치(7) - 손목(9)
left_arm_angle = calc_angle(xy[5], xy[7], xy[9])
print(f"왼쪽 팔꿈치 각도: {left_arm_angle:.1f}°")

# 오른쪽 팔: 어깨(6) - 팔꿈치(8) - 손목(10)
right_arm_angle = calc_angle(xy[6], xy[8], xy[10])
print(f"오른쪽 팔꿈치 각도: {right_arm_angle:.1f}°")

# 왼쪽 무릎: 엉덩이(11) - 무릎(13) - 발목(15)
left_knee_angle = calc_angle(xy[11], xy[13], xy[15])
print(f"왼쪽 무릎 각도: {left_knee_angle:.1f}°")

# 오른쪽 무릎: 엉덩이(12) - 무릎(14) - 발목(16)
right_knee_angle = calc_angle(xy[12], xy[14], xy[16])
print(f"오른쪽 무릎 각도: {right_knee_angle:.1f}°")

### 2-3. 키포인트 시각화 (직접 그리기)

ultralytics의 기본 시각화 대신, 키포인트를 직접 그려봅니다.

In [ ]:
# 뼈대(skeleton) 연결 정의
SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),          # 얼굴
    (5, 6),                                    # 어깨
    (5, 7), (7, 9),                            # 왼팔
    (6, 8), (8, 10),                           # 오른팔
    (5, 11), (6, 12),                          # 몸통
    (11, 12),                                  # 엉덩이
    (11, 13), (13, 15),                        # 왼다리
    (12, 14), (14, 16),                        # 오른다리
]

img = pose_r.orig_img.copy()
xy = kpts.xy[0].cpu().numpy()
conf = kpts.conf[0].cpu().numpy()

# 뼈대 그리기
for (i, j) in SKELETON:
    if conf[i] > 0.5 and conf[j] > 0.5:
        pt1 = tuple(xy[i].astype(int))
        pt2 = tuple(xy[j].astype(int))
        cv2.line(img, pt1, pt2, (0, 255, 255), 2)

# 관절 점 그리기
for i, ((x, y_coord), c) in enumerate(zip(xy, conf)):
    if c > 0.5:
        cv2.circle(img, (int(x), int(y_coord)), 5, (0, 0, 255), -1)
        cv2.putText(img, str(i), (int(x) + 5, int(y_coord) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Custom Skeleton Visualization")
plt.axis("off")
plt.show()

### 2-4. 웹캠 Pose (Colab)

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def capture_image():
    """브라우저 웹캠에서 이미지 1장을 캡처합니다."""
    js = Javascript('''
    async function captureImage() {
        const video = document.createElement('video');
        const stream = await navigator.mediaDevices.getUserMedia({ video: true });
        video.srcObject = stream;
        await new Promise((resolve) => { video.onloadedmetadata = () => resolve(); });
        video.play();
        await new Promise((resolve) => setTimeout(resolve, 1000));
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
        stream.getVideoTracks()[0].stop();
        return canvas.toDataURL('image/jpeg', 0.8);
    }
    captureImage();
    ''')
    display(js)
    data = eval_js('captureImage()')
    return data

def decode_image(data):
    """base64 데이터를 OpenCV 이미지로 변환합니다."""
    img_str = b64decode(data.split(',')[1])
    np_arr = np.frombuffer(img_str, np.uint8)
    return cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

# 웹캠 캡처 → Pose 추론
data = capture_image()
webcam_img = decode_image(data)

webcam_results = pose_model(webcam_img)
webcam_plot = webcam_results[0].plot()

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(webcam_plot, cv2.COLOR_BGR2RGB))
plt.title("Webcam Pose Estimation")
plt.axis("off")
plt.show()

# 관절 각도 출력
if webcam_results[0].keypoints is not None and len(webcam_results[0].keypoints) > 0:
    wxy = webcam_results[0].keypoints.xy[0].cpu().numpy()
    print(f"\n왼쪽 팔꿈치 각도: {calc_angle(wxy[5], wxy[7], wxy[9]):.1f}°")
    print(f"오른쪽 팔꿈치 각도: {calc_angle(wxy[6], wxy[8], wxy[10]):.1f}°")
    print(f"왼쪽 무릎 각도: {calc_angle(wxy[11], wxy[13], wxy[15]):.1f}°")
    print(f"오른쪽 무릎 각도: {calc_angle(wxy[12], wxy[14], wxy[16]):.1f}°")

In [ ]:
# 메모리 정리
del pose_model, pose_results
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 3. 3가지 태스크 한눈에 비교

같은 이미지(zidane.jpg)에 Detection, Segmentation, Pose를 동시에 적용합니다.

In [ ]:
test_img = "test_images/zidane.jpg"

models = {
    "Detection": YOLO("yolo26n.pt"),
    "Segmentation": YOLO("yolo26n-seg.pt"),
    "Pose": YOLO("yolo26n-pose.pt"),
}

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for ax, (name, model) in zip(axes, models.items()):
    results = model(test_img)
    plot = results[0].plot()
    ax.imshow(cv2.cvtColor(plot, cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=18)
    ax.axis("off")

plt.suptitle("같은 이미지 — 3가지 태스크 비교", fontsize=22, y=1.02)
plt.tight_layout()
plt.show()

# 정리
del models
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 4. (선택) OBB — 회전된 경계 상자

OBB(Oriented Bounding Box)는 기울어진 객체를 정확하게 탐지합니다.

프리트레인 모델은 **DOTA-v1** 데이터셋(위성/항공 영상)의 15개 클래스로 학습되어 있습니다:
비행기, 선박, 탱크, 야구장, 테니스코트, 농구코트, 항구, 다리, 차량, 헬리콥터, 원형교차로, 축구장, 수영장, 소형차, 대형차

In [ ]:
# DOTA 샘플 이미지 다운로드 (위성 영상)
obb_url = "https://ultralytics.com/images/boats.jpg"
obb_path = "test_images/boats.jpg"
if not os.path.exists(obb_path):
    urllib.request.urlretrieve(obb_url, obb_path)

obb_model = YOLO("yolo26n-obb.pt")
obb_results = obb_model(obb_path)

obb_plot = obb_results[0].plot()
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(obb_plot, cv2.COLOR_BGR2RGB))
plt.title("OBB: Oriented Bounding Boxes", fontsize=16)
plt.axis("off")
plt.show()

In [ ]:
# OBB 결과 데이터 확인
obb_r = obb_results[0]
if obb_r.obb is not None and len(obb_r.obb) > 0:
    print(f"탐지된 객체 수: {len(obb_r.obb)}")
    print(f"\nobb.xywhr shape: {obb_r.obb.xywhr.shape}")
    print(f"→ (x_center, y_center, width, height, rotation_radian)\n")

    print(f"{'객체':>10s} {'신뢰도':>8s} {'회전각도':>10s}")
    print("-" * 35)
    for xywhr, conf, cls in zip(
        obb_r.obb.xywhr.cpu().numpy(),
        obb_r.obb.conf.cpu().numpy(),
        obb_r.obb.cls.cpu().numpy().astype(int)
    ):
        angle_deg = np.degrees(xywhr[4])
        print(f"{obb_r.names[cls]:>10s} {conf:>8.2f} {angle_deg:>9.1f}°")
else:
    print("OBB 탐지 결과가 없습니다. (이 이미지는 DOTA 도메인이 아닐 수 있습니다)")

### Detection vs OBB 비교

In [ ]:
# 같은 이미지에 일반 Detection과 OBB를 비교
det_model2 = YOLO("yolo26n.pt")
det_results2 = det_model2(obb_path)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

det_plot2 = det_results2[0].plot()
axes[0].imshow(cv2.cvtColor(det_plot2, cv2.COLOR_BGR2RGB))
axes[0].set_title("Detection (수평 박스)", fontsize=16)
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(obb_plot, cv2.COLOR_BGR2RGB))
axes[1].set_title("OBB (회전 박스)", fontsize=16)
axes[1].axis("off")

plt.suptitle("기울어진 객체: Detection vs OBB", fontsize=20, y=1.02)
plt.tight_layout()
plt.show()

del det_model2, obb_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 5. (선택) Classification — CNN 분류와 비교

YOLO-cls는 ImageNet 1000 클래스로 프리트레인 되어 있어 별도 학습 없이 이미지를 분류합니다.

이전 CNN 수업에서 배운 분류와 동일한 태스크입니다.

In [ ]:
cls_model = YOLO("yolo26n-cls.pt")
cls_results = cls_model("test_images/bus.jpg")

cls_r = cls_results[0]
probs = cls_r.probs

print("=== Classification 결과 ===")
print(f"\n상위 5개 예측:")
for idx, conf in zip(probs.top5, probs.top5conf.cpu().numpy()):
    print(f"  {cls_r.names[idx]:<30s} {conf:.4f}")

print(f"\n→ Detection은 '어디에 무엇이 있는가'를 알려주지만,")
print(f"  Classification은 '이 이미지 전체가 무엇인가'만 알려줍니다.")

del cls_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 6. 태스크 선택 가이드

### 의사결정 플로우

```
이미지가 들어왔다!
    │
    ├─ "이 이미지가 뭔지만 알면 된다" → Classification
    │
    ├─ "어디에 뭐가 있는지 알아야 한다"
    │       │
    │       ├─ "위치만 알면 된다" → Detection
    │       ├─ "정확한 윤곽이 필요하다" → Segmentation
    │       ├─ "객체가 기울어져 있다" → OBB
    │       └─ "사람의 자세를 알아야 한다" → Pose
    │
    └─ "객체의 면적/비율을 계산해야 한다" → Segmentation
```

### 실전 퀴즈: 어떤 태스크를 선택할까?

아래 시나리오에 가장 적합한 태스크를 생각해 보세요.

1. 마트에서 진열대 제품 개수 세기
2. CT 영상에서 종양 크기 측정
3. 스포츠 선수 부상 방지 모니터링
4. 드론으로 찍은 주차장 차량 방향 분석
5. 품질 검사 라인에서 불량품/양품 구분

In [ ]:
# 정답 확인 (실행하세요)
answers = {
    "1. 마트 재고 관리": ("Detection", "위치와 개수만 파악하면 됨"),
    "2. 종양 크기 측정": ("Segmentation", "정확한 면적(픽셀 수)이 필요"),
    "3. 선수 부상 방지": ("Pose", "관절 각도로 자세 이상 감지"),
    "4. 차량 방향 분석": ("OBB", "다양한 각도로 주차된 차량의 방향 필요"),
    "5. 불량품 구분": ("Classification", "이미지 전체를 양품/불량으로 분류"),
}

for scenario, (task, reason) in answers.items():
    print(f"{scenario}")
    print(f"  → {task}: {reason}\n")

---
## 7. 최종 비교 요약

| | Detection | Segmentation | Classification | Pose | OBB |
|---|---|---|---|---|---|
| **출력** | 박스 | 마스크 | 클래스 | 키포인트 | 회전 박스 |
| **대상** | 모든 객체 | 모든 객체 | 이미지 전체 | 사람만 | 기울어진 객체 |
| **모델** | `yolo26n.pt` | `yolo26n-seg.pt` | `yolo26n-cls.pt` | `yolo26n-pose.pt` | `yolo26n-obb.pt` |
| **데이터** | COCO 80cls | COCO 80cls | ImageNet 1000cls | COCO keypoints | DOTA-v1 15cls |
| **핵심** | 위치+클래스 | 윤곽+클래스 | 클래스 | 관절 좌표 | 위치+방향 |

---
## 심화 과제

1. Segmentation 마스크로 객체별 면적(픽셀 수)을 계산하고, 가장 큰 객체를 찾아보세요
2. Pose 키포인트로 "손을 든 사람"을 자동 감지하는 조건문을 만들어 보세요 (힌트: 손목 y좌표 < 어깨 y좌표)
3. 자신의 이미지/영상으로 5가지 태스크를 모두 실행하고 결과를 비교해 보세요
4. nano(n) 모델과 small(s) 모델의 속도 차이를 `%%timeit`으로 측정해 보세요